# F-beta Score

This notebook illustrates why the $F_\beta$ score becomes more sensitive to recall when $\beta > 1$ and more sensitive to precision when $\beta < 1$. Here's what the code does:

1. **Defines the $F_\beta$ score**
   - Implements the formula for arbitrary precision, recall, and $\beta$
   - Uses a numerically safe version to avoid division-by-zero problems

2. **Fixes one metric and varies the other**
   - Keeps precision fixed and varies recall
   - Keeps recall fixed and varies precision
   - Compares the result for different values of $\beta$

3. **Visualizes the full surface**
   - Creates heatmaps of $F_\beta(\text{precision}, \text{recall})$
   - Compares a case that favors precision ($\beta = 0.5$) with a case that favors recall ($\beta = 2$)

4. **Builds intuition with concrete examples**
   - Compares two hypothetical classifiers
   - Shows how the preferred model can change when $\beta$ changes

When you run this code, you'll see:
- Why $\beta > 1$ rewards recall improvements more strongly
- Why $\beta < 1$ rewards precision improvements more strongly
- How the same pair of models can be ranked differently depending on the application


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Define the F-beta score
def f_beta(precision, recall, beta=1.0):
    """Compute the F-beta score for scalar or array inputs."""
    beta2 = beta ** 2
    numerator = (1 + beta2) * precision * recall
    denominator = beta2 * precision + recall
    return np.divide(numerator, denominator, out=np.zeros_like(numerator, dtype=float), where=denominator > 0)

# Values used in the comparisons
betas = [0.5, 1.0, 2.0, 4.0]
recall_values = np.linspace(0.01, 1.0, 300)
precision_values = np.linspace(0.01, 1.0, 300)
fixed_precision = 0.80
fixed_recall = 0.80

# Grid for heatmaps
P, R = np.meshgrid(np.linspace(0.01, 1.0, 200), np.linspace(0.01, 1.0, 200))
F_beta_05 = f_beta(P, R, beta=0.5)
F_beta_2 = f_beta(P, R, beta=2.0)

# Visualize how beta changes the sensitivity to precision and recall
plt.figure(figsize=(14, 10))

# Fix precision and vary recall
plt.subplot(2, 2, 1)
for beta in betas:
    scores = f_beta(fixed_precision, recall_values, beta=beta)
    plt.plot(recall_values, scores, lw=2, label=f'beta = {beta}')

plt.xlabel('Recall')
plt.ylabel('F-beta score')
plt.title(f'Fixed Precision = {fixed_precision:.2f}, Varying Recall')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# Fix recall and vary precision
plt.subplot(2, 2, 2)
for beta in betas:
    scores = f_beta(precision_values, fixed_recall, beta=beta)
    plt.plot(precision_values, scores, lw=2, label=f'beta = {beta}')

plt.xlabel('Precision')
plt.ylabel('F-beta score')
plt.title(f'Fixed Recall = {fixed_recall:.2f}, Varying Precision')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# Heatmap for beta < 1
plt.subplot(2, 2, 3)
im1 = plt.imshow(F_beta_05, origin='lower', extent=[0.01, 1.0, 0.01, 1.0], aspect='auto', cmap='YlOrRd')
plt.colorbar(im1, fraction=0.046, pad=0.04)
plt.xlabel('Precision')
plt.ylabel('Recall')
plt.title('F-beta Surface for beta = 0.5')

# Heatmap for beta > 1
plt.subplot(2, 2, 4)
im2 = plt.imshow(F_beta_2, origin='lower', extent=[0.01, 1.0, 0.01, 1.0], aspect='auto', cmap='YlGnBu')
plt.colorbar(im2, fraction=0.046, pad=0.04)
plt.xlabel('Precision')
plt.ylabel('Recall')
plt.title('F-beta Surface for beta = 2.0')

plt.tight_layout()

# Compare hypothetical classifiers
model_a = {'precision': 0.90, 'recall': 0.60}
model_b = {'precision': 0.70, 'recall': 0.85}

print('Comparison of hypothetical classifiers:')
print('-' * 60)
print(f"Model A: precision = {model_a['precision']:.2f}, recall = {model_a['recall']:.2f}")
print(f"Model B: precision = {model_b['precision']:.2f}, recall = {model_b['recall']:.2f}")

for beta in betas:
    score_a = f_beta(model_a['precision'], model_a['recall'], beta=beta)
    score_b = f_beta(model_b['precision'], model_b['recall'], beta=beta)
    preferred = 'A' if score_a > score_b else 'B'
    print(f"\nFor beta = {beta}:")
    print(f"  F_beta(Model A) = {score_a:.4f}")
    print(f"  F_beta(Model B) = {score_b:.4f}")
    print(f"  Preferred model: {preferred}")

print('\nInterpretation:')
print('-' * 60)
print('When beta increases, recall receives relatively more weight.')
print('When beta decreases, precision receives relatively more weight.')
print('The factor beta^2 is the reason this change can become quite strong.')

plt.show()
